# AlphaGenome Genomic Prediction

![AlphaGenome Genomic Prediction](https://proto-bio.github.io/proto-assets/images/tool/alphagenome/hero.png)

AlphaGenome is a multi-task genomic foundation model from Google DeepMind that predicts regulatory signals, splicing, and 3D contact maps from DNA sequence at single base-pair resolution. It supports multiple context lengths (16 Kb to 1 Mb) and provides six tool functions covering interval prediction, variant-effect prediction, raw sequence prediction, variant scoring, interval scoring, and in-silico saturating mutagenesis.

This notebook demonstrates four representative functions: interval prediction, variant-effect prediction, variant scoring, and in-silico mutagenesis. The remaining two functions (`run_alphagenome_predict_sequences` and `run_alphagenome_score_intervals`) follow the same patterns and share input schemas with the functions shown here.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("alphagenome")
display_overview("alphagenome")
display_docs_section("alphagenome", "Background")

# AlphaGenome

AlphaGenome is Google DeepMind's multi-task genomic foundation model that predicts diverse regulatory signals from DNA sequence. It takes long genomic context windows (up to 1 Mb) and outputs spatial prediction tracks for [RNA-seq](https://en.wikipedia.org/wiki/RNA-Seq), [ATAC-seq](https://en.wikipedia.org/wiki/ATAC-seq), [DNase-seq](https://en.wikipedia.org/wiki/DNase-Seq), [CAGE](https://en.wikipedia.org/wiki/Cap_analysis_of_gene_expression), [ChIP-seq](https://en.wikipedia.org/wiki/ChIP_sequencing), splice sites, and 3D contact maps. This wrapper provides six batched tools covering interval prediction, variant-effect prediction, raw sequence prediction, and three scoring modes (variant, interval, ISM).

This implementation uses **local GPU inference** with Hugging Face weights through an isolated standalone venv runtime (`ToolInstance("alphagenome")`).

## Background

Gene regulation is controlled by a complex interplay of [cis-regulatory elements](https://en.wikipedia.org/wiki/Cis-regulatory_element) (promoters, enhancers, silencers) and [chromatin](https://en.wikipedia.org/wiki/Chromatin) state. Different experimental assays measure different aspects of this regulatory landscape:

- **RNA-seq** measures transcript abundance, reflecting gene expression levels
- **ATAC-seq / DNase-seq** measure chromatin accessibility, indicating active regulatory regions
- **CAGE** captures transcription start sites at single-nucleotide resolution
- **ChIP-seq** maps protein-DNA interactions (histone modifications, transcription factor binding)
- **Splice site assays** quantify alternative splicing patterns
- **Contact maps ([Hi-C](https://en.wikipedia.org/wiki/Hi-C_(genomic_analysis_technique)))** capture 3D chromatin organization

AlphaGenome jointly predicts all of these signals from DNA sequence alone, leveraging long-range context (up to 1 Mb) to capture distal regulatory effects. This multi-task architecture allows it to model the relationships between regulatory layers -- for example, how a variant that disrupts a [CTCF](https://en.wikipedia.org/wiki/CTCF) binding site might alter both chromatin accessibility and 3D contact structure.

## Available tools

In [2]:
display_available_tools("alphagenome")

- **`run_alphagenome_predict_intervals()`** — Predict genomic signals for batched intervals using AlphaGenome
- **`run_alphagenome_predict_sequences()`** — Predict genomic signals from batched raw DNA sequences using AlphaGenome
- **`run_alphagenome_predict_variants()`** — Predict variant effects in batch using AlphaGenome
- **`run_alphagenome_score_intervals()`** — Score genomic intervals in batch with AlphaGenome interval scorers
- **`run_alphagenome_score_ism_variants_batch()`** — Run batched in-silico mutagenesis with AlphaGenome variant scorers
- **`run_alphagenome_score_variants()`** — Score variant effects in batch with AlphaGenome variant scorers

### `run_alphagenome_predict_intervals`

Given a genomic interval (chromosome, start, end), this function runs the AlphaGenome model and returns multi-track regulatory predictions for that region. The model supports RNA-seq, ATAC, DNase, histone ChIP, transcription factor ChIP, splice site, and contact map outputs. The interval is automatically resized to the nearest supported context length if needed. Output tracks can be filtered by ontology term (cell line or tissue) to focus on relevant biological contexts.

In [3]:
from proto_tools import (
    AlphaGenomeInterval,
    AlphaGenomePredictIntervalsConfig,
    AlphaGenomePredictIntervalsInput,
    run_alphagenome_predict_intervals,
)

In [4]:
# Display input docs
display_api_reference("alphagenome", "input", "run_alphagenome_predict_intervals")

# Paper Figure 2a coordinates: chr19, 1 Mb window, HepG2 cell line
interval_inputs = AlphaGenomePredictIntervalsInput(
    intervals=AlphaGenomeInterval(
        chromosome="chr19",
        interval_start=10_587_331,
        interval_end=11_635_907,
    ),
)

**Input** — `AlphaGenomePredictIntervalsInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `intervals` | `list[proto_tools.tools.sequence_scoring.alphagenome.shared_data_models.AlphaGenomeInterval]` | required | Genomic intervals for prediction |

In [5]:
# Display config docs
display_api_reference("alphagenome", "config", "run_alphagenome_predict_intervals")

interval_config = AlphaGenomePredictIntervalsConfig(
    requested_outputs=["RNA_SEQ", "ATAC", "DNASE", "CHIP_HISTONE", "CHIP_TF"],
    ontology_terms=["EFO:0001187"],  # HepG2
    organism="human",
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `AlphaGenomePredictConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run AlphaGenome inference on |
| `timeout` | `int` | `1800` | Maximum execution time in seconds (AlphaGenome JAX compilation is slow on first run) |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `model_version` | `str` | `'all_folds'` | AlphaGenome Hugging Face model version |
| `requested_outputs` | `list[Literal['ATAC', 'CAGE', 'DNASE', 'RNA_SEQ', 'CHIP_HISTONE', 'CHIP_TF', 'SPLICE_SITES', 'SPLICE_SITE_USAGE', 'SPLICE_JUNCTIONS', 'CONTACT_MAPS', 'PROCAP']]` | `['RNA_SEQ']` | Output type names to request from AlphaGenome |
| `ontology_terms` | `list[str] \| None` | `None` | UBERON tissue/cell IDs to filter tracks (e.g. 'UBERON:0001167'); None = all |
| `organism` | `Literal['human', 'mouse']` | `'human'` | Organism for AlphaGenome predictions |

In [6]:
# Run the interval prediction function
interval_result = run_alphagenome_predict_intervals(interval_inputs, interval_config)

Running run_alphagenome_predict_intervals [00:00]

In [7]:
display_api_reference("alphagenome", "output", "run_alphagenome_predict_intervals")

output = interval_result[0]
print(f"Requested outputs: {output.requested_outputs}")
print(f"Result keys: {list(output.result.get('predictions', {}).keys())}")

**Output** — `AlphaGenomePredictIntervalsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `list[proto_tools.tools.sequence_scoring.alphagenome.shared_data_models.AlphaGenomePredictOutput]` | required | Per-interval AlphaGenome prediction outputs |

Requested outputs: ['RNA_SEQ', 'ATAC', 'DNASE', 'CHIP_HISTONE', 'CHIP_TF']
Result keys: ['atac', 'cage', 'dnase', 'rna_seq', 'chip_histone', 'chip_tf', 'splice_sites', 'splice_site_usage', 'splice_junctions', 'contact_maps', 'procap']


### `run_alphagenome_predict_variants`

Given a variant (chromosome, context window, variant position, reference and alternate alleles), this function runs AlphaGenome twice — once for the reference sequence and once for the alternate — and returns the full track-level predictions for both alleles. This is the raw prediction function that gives access to the complete output tensor for each allele. It reproduces Figure 3c from the AlphaGenome paper: a G-to-C variant at chr21:46,126,238 in the COL6A2 gene that causes alternative splice junction formation.

In [8]:
from proto_tools import (
    AlphaGenomePredictVariantsConfig,
    AlphaGenomePredictVariantsInput,
    AlphaGenomeVariant,
    run_alphagenome_predict_variants,
)

In [9]:
# Display input docs
display_api_reference("alphagenome", "input", "run_alphagenome_predict_variants")

# Paper Figure 3c coordinates: chr21:46,126,238 G>C (COL6A2), 1 Mb context, Aorta tissue
variant_inputs = AlphaGenomePredictVariantsInput(
    variants=AlphaGenomeVariant(
        chromosome="chr21",
        interval_start=45_601_950,
        interval_end=46_650_526,
        variant_position=46_126_238,
        reference_bases="G",
        alternate_bases="C",
    ),
)

**Input** — `AlphaGenomePredictVariantsInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `variants` | `list[proto_tools.tools.sequence_scoring.alphagenome.shared_data_models.AlphaGenomeVariant]` | required | Variants (with intervals) for prediction |

In [10]:
# Display config docs
display_api_reference("alphagenome", "config", "run_alphagenome_predict_variants")

variant_config = AlphaGenomePredictVariantsConfig(
    requested_outputs=["RNA_SEQ", "SPLICE_SITE_USAGE"],
    ontology_terms=["UBERON:0001496"],  # Aorta
    organism="human",
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `AlphaGenomePredictConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run AlphaGenome inference on |
| `timeout` | `int` | `1800` | Maximum execution time in seconds (AlphaGenome JAX compilation is slow on first run) |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `model_version` | `str` | `'all_folds'` | AlphaGenome Hugging Face model version |
| `requested_outputs` | `list[Literal['ATAC', 'CAGE', 'DNASE', 'RNA_SEQ', 'CHIP_HISTONE', 'CHIP_TF', 'SPLICE_SITES', 'SPLICE_SITE_USAGE', 'SPLICE_JUNCTIONS', 'CONTACT_MAPS', 'PROCAP']]` | `['RNA_SEQ']` | Output type names to request from AlphaGenome |
| `ontology_terms` | `list[str] \| None` | `None` | UBERON tissue/cell IDs to filter tracks (e.g. 'UBERON:0001167'); None = all |
| `organism` | `Literal['human', 'mouse']` | `'human'` | Organism for AlphaGenome predictions |

In [11]:
# Run the variant-effect prediction function
variant_result = run_alphagenome_predict_variants(variant_inputs, variant_config)

Running run_alphagenome_predict_variants [00:00]

In [12]:
display_api_reference("alphagenome", "output", "run_alphagenome_predict_variants")

output = variant_result[0]
print(f"Variant metadata: {output.variant}")
print(f"Result keys: {list(output.result.get('predictions', {}).keys())}")

**Output** — `AlphaGenomePredictVariantsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `list[proto_tools.tools.sequence_scoring.alphagenome.shared_data_models.AlphaGenomePredictOutput]` | required | Per-variant AlphaGenome prediction outputs |

Variant metadata: {'position': 46126238, 'reference_bases': 'G', 'alternate_bases': 'C'}
Result keys: ['reference', 'alternate']


### `run_alphagenome_score_variants`

This function evaluates variants using AlphaGenome's recommended variant scorers and returns a tidy table of effect-size scores with one row per scorer-track-gene combination. It is more convenient than raw variant-effect prediction when you need quantitative effect sizes rather than full track-level predictions. The same COL6A2 variant from the previous section is used here to demonstrate the scoring output format.

In [13]:
import pandas as pd
from proto_tools import (
    AlphaGenomeScoreVariantsConfig,
    AlphaGenomeScoreVariantsInput,
    run_alphagenome_score_variants,
)

In [14]:
# Display input docs
display_api_reference("alphagenome", "input", "run_alphagenome_score_variants")

score_variant_inputs = AlphaGenomeScoreVariantsInput(
    variants=AlphaGenomeVariant(
        chromosome="chr21",
        interval_start=45_601_950,
        interval_end=46_650_526,
        variant_position=46_126_238,
        reference_bases="G",
        alternate_bases="C",
    ),
)

**Input** — `AlphaGenomeScoreVariantsInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `variants` | `list[proto_tools.tools.sequence_scoring.alphagenome.shared_data_models.AlphaGenomeVariant]` | required | Variants (with intervals) for scoring |

In [15]:
# Display config docs
display_api_reference("alphagenome", "config", "run_alphagenome_score_variants")

score_variant_config = AlphaGenomeScoreVariantsConfig(
    variant_scorers=["RNA_SEQ", "SPLICE_SITE_USAGE"],
    organism="human",
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `AlphaGenomeScoreVariantsConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run AlphaGenome inference on |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `model_version` | `str` | `'all_folds'` | AlphaGenome Hugging Face model version |
| `variant_scorers` | `list[Literal['ATAC', 'CONTACT_MAPS', 'DNASE', 'CHIP_TF', 'CHIP_HISTONE', 'CAGE', 'PROCAP', 'RNA_SEQ', 'RNA_SEQ_ACTIVE', 'SPLICE_SITES', 'SPLICE_SITE_USAGE', 'SPLICE_JUNCTIONS', 'POLYADENYLATION', 'ATAC_ACTIVE', 'DNASE_ACTIVE', 'CHIP_TF_ACTIVE', 'CHIP_HISTONE_ACTIVE', 'CAGE_ACTIVE', 'PROCAP_ACTIVE']] \| None` | `None` | Scorer names to use. None uses all recommended scorers. |
| `organism` | `Literal['human', 'mouse']` | `'human'` | Organism for AlphaGenome predictions |

In [16]:
# Run the variant scoring function
score_variant_result = run_alphagenome_score_variants(score_variant_inputs, score_variant_config)

Running run_alphagenome_score_variants [00:00]

In [17]:
display_api_reference("alphagenome", "output", "run_alphagenome_score_variants")

output = score_variant_result[0]
print(f"Returned {len(output.scores)} score records")
if output.scores:
    df = pd.DataFrame(output.scores)
    print(df.head(10).to_string())

**Output** — `AlphaGenomeScoreVariantsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `list[proto_tools.tools.sequence_scoring.alphagenome.shared_data_models.AlphaGenomeScoreOutput]` | required | Per-variant AlphaGenome score outputs |

Returned 15415 score records
                                                                                                                                                                                                                                                                                                                                                                                                                                                  variant_id                                                                                                                                 scored_interval          gene_id        gene_name gene_type gene_strand junction_Start junction_End output_type                               variant_scorer                     track_name track_strand         Assay title ontology_curie          biosample_name                 biosample_type biosample_life_stage gtex_tissue data_source endedness genetically_modified  raw_score
0  {'alternate_bases

### `run_alphagenome_score_ism_variants_batch`

In-silico saturating mutagenesis (ISM) systematically mutates every position within a sub-interval to every possible alternate base and scores each substitution. The result is a comprehensive map of positional sensitivity that reveals which bases within a regulatory element are critical for its function. This example saturate-mutates a 10 bp window around the COL6A2 variant, scoring each single-nucleotide substitution with the splice sites scorer.

In [18]:
from proto_tools import (
    AlphaGenomeISM,
    AlphaGenomeScoreISMConfig,
    AlphaGenomeScoreISMInput,
    run_alphagenome_score_ism_variants_batch,
)

In [19]:
# Display input docs
display_api_reference("alphagenome", "input", "run_alphagenome_score_ism_variants_batch")

ism_inputs = AlphaGenomeScoreISMInput(
    requests=AlphaGenomeISM(
        chromosome="chr21",
        interval_start=45_601_950,
        interval_end=46_650_526,
        ism_interval_start=46_126_233,
        ism_interval_end=46_126_243,  # 10 bp window around variant
    ),
)

**Input** — `AlphaGenomeScoreISMInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `requests` | `list[proto_tools.tools.sequence_scoring.alphagenome.alphagenome_score_ism_variants_batch.AlphaGenomeISM]` | required | ISM requests to process |

In [20]:
# Display config docs
display_api_reference("alphagenome", "config", "run_alphagenome_score_ism_variants_batch")

ism_config = AlphaGenomeScoreISMConfig(
    variant_scorers=["SPLICE_SITE_USAGE"],
    organism="human",
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `AlphaGenomeScoreVariantsConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run AlphaGenome inference on |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `model_version` | `str` | `'all_folds'` | AlphaGenome Hugging Face model version |
| `variant_scorers` | `list[Literal['ATAC', 'CONTACT_MAPS', 'DNASE', 'CHIP_TF', 'CHIP_HISTONE', 'CAGE', 'PROCAP', 'RNA_SEQ', 'RNA_SEQ_ACTIVE', 'SPLICE_SITES', 'SPLICE_SITE_USAGE', 'SPLICE_JUNCTIONS', 'POLYADENYLATION', 'ATAC_ACTIVE', 'DNASE_ACTIVE', 'CHIP_TF_ACTIVE', 'CHIP_HISTONE_ACTIVE', 'CAGE_ACTIVE', 'PROCAP_ACTIVE']] \| None` | `None` | Scorer names to use. None uses all recommended scorers. |
| `organism` | `Literal['human', 'mouse']` | `'human'` | Organism for AlphaGenome predictions |

In [21]:
# Run the ISM scoring function
ism_result = run_alphagenome_score_ism_variants_batch(ism_inputs, ism_config)

Running run_alphagenome_score_ism_variants_batch [00:00]

In [22]:
display_api_reference("alphagenome", "output", "run_alphagenome_score_ism_variants_batch")

output = ism_result[0]
print(f"Returned {len(output.scores)} score records")
if output.scores:
    df = pd.DataFrame(output.scores)
    print(df.head(10).to_string())

**Output** — `AlphaGenomeScoreISMOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `list[proto_tools.tools.sequence_scoring.alphagenome.shared_data_models.AlphaGenomeScoreOutput]` | required | Per-request AlphaGenome ISM score outputs |

Returned 11010 score records
                                                                                                                                                                                                                                                                                                                                                                                                                                                  variant_id                                                                                                                                 scored_interval          gene_id gene_name       gene_type gene_strand junction_Start junction_End        output_type                                                          variant_scorer                           track_name track_strand         Assay title ontology_curie         biosample_name                 biosample_type biosample_life_stage gtex_tissue data_source  raw_score
0  {'alternat

## Export Results

Prediction and scoring outputs can be saved to standard file formats for downstream analysis. Prediction outputs support JSON and NumPy formats, while scoring outputs support JSON and CSV. The export code below is shown for reference but not executed, as outputs can be very large.

In [23]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

interval_result[0].export("alphagenome_predict_intervals", export_path=output_dir, file_format="json")
variant_result[0].export("alphagenome_predict_variants", export_path=output_dir, file_format="json")
score_variant_result[0].export("alphagenome_score_variants", export_path=output_dir, file_format="csv")
ism_result[0].export("alphagenome_score_ism_variants_batch", export_path=output_dir, file_format="csv")